In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import date
from pathlib import Path
import time

BASE_URL = "https://www.gutenberg.org"
TOP_URL = f"{BASE_URL}/browse/scores/top"

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = DATA_DIR / "weekly_rankings.csv"


def fetch_top_page():
    headers = {
        "User-Agent": "gutenberg-weekly-trends/1.0"
    }
    response = requests.get(TOP_URL, headers=headers, timeout=30)
    response.raise_for_status()
    return response.text


def extract_top_10(html):
    soup = BeautifulSoup(html, "html.parser")

    # Find the header for "Top 100 EBooks last 7 days"
    header = soup.find("h2", string="Top 100 EBooks last 7 days")
    book_list = header.find_next_sibling("ol")

    top_books = []
    for rank, li in enumerate(book_list.find_all("li")[:10], start=1):
        link = li.find("a")
        book_id = int(link["href"].split("/")[-1])

        top_books.append({
            "week_date": date.today().isoformat(),
            "rank": rank,
            "book_id": book_id,
            "title": link.text.strip()
        })

    return top_books


def save_results(records):
    df = pd.DataFrame(records)

    if OUTPUT_FILE.exists():
        df_existing = pd.read_csv(OUTPUT_FILE)
        df = pd.concat([df_existing, df], ignore_index=True)

    df.to_csv(OUTPUT_FILE, index=False)


def main():
    print("Fetching Gutenberg weekly rankings...")
    html = fetch_top_page()
    time.sleep(2)

    records = extract_top_10(html)
    save_results(records)

    print(f"Saved {len(records)} records to {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Fetching Gutenberg weekly rankings...
Saved 10 records to data\raw\weekly_rankings.csv
